# Gold layer
## What the layer contains and business questions
- Table 1: 
    - Grain: one row per fault_id × variable, 
    - Columns: fault-free baseline mean, post-fault mean, absolute deviation, % deviation
    - Answers: "How does each fault affect each variable?" — your 20-table idea, collapsed into one
- Table 2: 
    - Grain: one row per variable
    - Columns: mean, std, UCL/LCL at ±2σ and ±3σ, is_normal, description, unit
    - Answers: "What are the expected operating ranges and where should alerts trigger?" — your variability/AC table

## First
Confirm Silver Tabels exist and restate constants

In [0]:
%sql
SHOW TABLES IN tep_lakehouse.silver;

database,tableName,isTemporary
silver,faultfree_normality_stats,false
silver,faultfree_normality_stats_enriched,false
silver,faultfree_testing,false
silver,faultfree_training,false
silver,faulty_testing_fault_01,false
silver,faulty_testing_fault_02,false
silver,faulty_testing_fault_03,false
silver,faulty_testing_fault_04,false
silver,faulty_testing_fault_05,false
silver,faulty_testing_fault_06,false


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd

VAR_COLS  = [f"xmeas_{i}" for i in range(1, 42)] + [f"xmv_{i}" for i in range(1, 12)]
FAULT_IDS = list(range(1, 21))

spark.sql("CREATE SCHEMA IF NOT EXISTS tep_lakehouse.gold")

DataFrame[]

## Second
Create the control limits table with baseline + UCL/LCL

In [0]:
df_stats = spark.table("tep_lakehouse.silver.faultfree_normality_stats")
df_ref   = spark.table("tep_lakehouse.silver.variable_reference")

df_gold = (
    df_stats
    .join(df_ref, on="variable", how="left")
    .withColumn("ucl_2sigma", F.col("mean") + 2 * F.col("std"))
    .withColumn("lcl_2sigma", F.col("mean") - 2 * F.col("std"))
    .withColumn("ucl_3sigma", F.col("mean") + 3 * F.col("std"))
    .withColumn("lcl_3sigma", F.col("mean") - 3 * F.col("std"))
    .select(
        "variable", "description", "unit", "type", "category", "location",
        "mean", "std", "skewness", "kurtosis",
        "lcl_3sigma", "lcl_2sigma", "ucl_2sigma", "ucl_3sigma",
        "is_normal", "shapiro_p"
    )
    .orderBy("type", "variable")
)

df_gold.write.mode("overwrite").saveAsTable("tep_lakehouse.gold.process_control_limits")
print(f"Written: tep_lakehouse.gold.process_control_limits ({df_gold.count()} rows)")

Written: tep_lakehouse.gold.process_control_limits (52 rows)


In [0]:
%sql
SELECT * FROM tep_lakehouse.gold.process_control_limits

variable,description,unit,type,category,location,mean,std,skewness,kurtosis,lcl_3sigma,lcl_2sigma,ucl_2sigma,ucl_3sigma,is_normal,shapiro_p
xmeas_1,A Feed,kscmh,XMEAS,flow,stream_1,0.2505263481666667,0.0013657179664077358,-0.08942561154419909,0.030323230412548696,0.24642919426744347,0.24779491223385122,0.2532577840994822,0.2546235020658899,false,0.011592436144534946
xmeas_10,Purge Rate,kscmh,XMEAS,flow,stream_9,0.33707008231249996,5.890589299225144E-4,0.19633125094766246,0.3020056199349779,0.3353029055227324,0.33589196445265496,0.33824820017234497,0.33883725910226753,false,0.003742035527842247
xmeas_11,Product Separator Temperature,°C,XMEAS,temperature,separator,80.10693857708334,0.011857156039429013,0.12457571225166866,-0.10113928208256384,80.07136710896505,80.08322426500449,80.13065288916219,80.14251004520163,true,0.17328431575632647
xmeas_12,Product Separator Level,%,XMEAS,level,separator,49.99956074791667,0.044593040300007554,-0.044510209905002,0.09998461702770367,49.865781627016645,49.91037466731665,50.08874682851668,50.13333986881669,true,0.9552944743194148
xmeas_13,Product Separator Pressure,kPa,XMEAS,pressure,separator,2633.741935,0.3942555069219198,-0.11195986780451278,-0.45177114909924887,2632.5591684792344,2632.9534239861564,2634.5304460138436,2634.9247015207657,false,1.8565090106837894E-4
xmeas_14,Product Separator Underflow,m3/hr,XMEAS,flow,stream_10,25.161281764583336,0.044985890151704476,0.02480598541694842,0.052837194986288694,25.026324094128224,25.071309984279928,25.251253544886744,25.296239435038448,true,0.6487361392179662
xmeas_15,Stripper Level,%,XMEAS,level,stripper,50.00006492916667,0.045987964460350035,-0.07519844439398814,-0.055213084820811,49.86210103578562,49.908089000245965,50.09204085808737,50.13802882254772,true,0.4668550147655847
xmeas_16,Stripper Pressure,kPa,XMEAS,pressure,stripper,3102.2159433333336,0.3239650596470721,-0.20187562713899646,-0.3276701683510157,3101.244048154392,3101.5680132140396,3102.8638734526276,3103.187838512275,false,2.319741199558739E-5
xmeas_17,Stripper Underflow,m3/hr,XMEAS,flow,stream_11,22.947002141666665,0.027259788381792993,-0.27957648003644897,0.16919836576900726,22.865222776521286,22.89248256490308,23.00152171843025,23.028781506812045,false,0.0017265636253408138
xmeas_18,Stripper Temperature,°C,XMEAS,temperature,stripper,65.80285650625,0.02486176445924697,-0.42878815802793496,1.0332287415477923,65.72827121287226,65.7531329773315,65.85258003516849,65.87744179962773,false,2.3243098384530596E-12


## Third
Per fault_id and variable we see the mean pre and post mean, the absolute and relative deviation

In [0]:
# Pull fault-free baseline means — one value per variable to compare against
df_baseline = (
    spark.table("tep_lakehouse.silver.faultfree_normality_stats")
    .select("variable", F.col("mean").alias("baseline_mean"))
)

# Convert to dict for lightweight lookup — only 52 rows
baseline = {row["variable"]: row["baseline_mean"] for row in df_baseline.collect()}

records = []

for fault_id in FAULT_IDS:
    df_fault = spark.table(f"tep_lakehouse.silver.faulty_training_fault_{fault_id:02d}").toPandas()
    
    # Only use post-fault samples
    df_post = df_fault[df_fault["time_since_fault_h"] >= 0]
    
    for var in VAR_COLS:
        post_mean = df_post[var].mean()
        base_mean = baseline[var]
        
        records.append({
            "fault_id":          fault_id,
            "variable":          var,
            "baseline_mean":     base_mean,
            "post_fault_mean":   post_mean,
            "abs_deviation":     abs(post_mean - base_mean),
            "pct_deviation":     abs(post_mean - base_mean) / abs(base_mean) * 100 if base_mean != 0 else None,
        })

df_ref = spark.table("tep_lakehouse.silver.variable_reference")

df_gold = (
    spark.createDataFrame(pd.DataFrame(records))
    .join(df_ref, on="variable", how="left")
    .select(
        "fault_id", "variable", "description", "unit", "type", "category",
        "baseline_mean", "post_fault_mean", "abs_deviation", "pct_deviation"
    )
    .orderBy("fault_id", F.col("pct_deviation").desc())
)

df_gold.write.mode("overwrite").saveAsTable("tep_lakehouse.gold.fault_response")
print(f"Written: tep_lakehouse.gold.fault_response ({df_gold.count()} rows)")
# Expected: 20 faults × 52 variables = 1040 rows

Written: tep_lakehouse.gold.fault_response (1040 rows)


In [0]:
%sql SELECT * FROM tep_lakehouse.gold.fault_response

fault_id,variable,description,unit,type,category,baseline_mean,post_fault_mean,abs_deviation,pct_deviation
1,xmv_3,A Feed Flow Valve,%,XMV,valve,24.643706191666663,74.48902244074843,49.84531624908176,202.26387971601892
1,xmeas_1,A Feed,kscmh,XMEAS,flow,0.2505263481666667,0.7572444594178794,0.5067181112512127,202.26140482202308
1,xmv_9,Stripper Steam Valve,%,XMV,valve,47.97524535833333,64.4217247983368,16.446479440003465,34.28117838098081
1,xmeas_19,Stripper Steam Flow,kg/hr,XMEAS,flow,232.21459906249999,285.4171156756757,53.2025166131757,22.910926715187436
1,xmeas_4,A and C Feed,kscmh,XMEAS,flow,9.346865773333333,8.79682797006237,0.5500378032709623,5.884729882825788
1,xmv_4,A and C Feed Flow Valve,%,XMV,valve,61.29568916666666,57.690704457380455,3.6049847092862066,5.88130218992079
1,xmeas_10,Purge Rate,kscmh,XMEAS,flow,0.33707008231249996,0.3180559628690228,0.01901411944347714,5.640998843038529
1,xmv_6,Purge Valve,%,XMV,valve,40.0579255375,37.88145911434511,2.176466423154885,5.433297890369781
1,xmeas_18,Stripper Temperature,°C,XMEAS,temperature,65.80285650625,68.32767650311851,2.5248199968685157,3.836945887947443
1,xmeas_28,Reactor Feed Component F,mol%,XMEAS,composition,1.6568011570833332,1.6875442856548857,0.03074312857155248,1.855571408802811
